# Phase 4: Custom Triton Kernels - LayerNorm

*(This notebook is part of the GPT-2 Inference Engine from Scratch project.)*

## Lesson 25: First Triton Kernel

**Goal:** We are writing our very first custom GPU kernel using OpenAI's Triton. It's a simple `y = x + 1` operation just to get familiar with Triton syntax, block pointers, and grid execution.

In [25]:
import triton
import triton.language as tl

@triton.jit
def add_one_kernel(x_ptr, y_ptr, n, BLOCK_SIZE: tl.constexpr):
    pid = tl.program_id(0)
    offsets = pid * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < n
    x = tl.load(x_ptr + offsets, mask=mask)
    tl.store(y_ptr + offsets, x + 1.0, mask=mask)

n = 1_000_000
x = torch.randn(n, device="cuda")
y = torch.empty_like(x)

BLOCK_SIZE = 1024
grid = (triton.cdiv(n, BLOCK_SIZE),)
add_one_kernel[grid](x, y, n, BLOCK_SIZE=BLOCK_SIZE)

y_ref = x + 1.0
print("Triton version:", triton.__version__)
print("Grid size (num blocks):", grid[0])
print("Exact match vs torch:", torch.equal(y, y_ref))

Triton version: 3.6.0
Grid size (num blocks): 977
Exact match vs torch: True


## Lesson 26: Triton vs PyTorch Bandwidth

**Goal:** Let's benchmark a 128M-element addition using both PyTorch and our Triton kernel. We want to see how much memory bandwidth (GB/s) we are actually able to squeeze out of the GPU.

In [26]:
import time

N = 128 * 1024 * 1024
x = torch.randn(N, device="cuda")
y_torch = torch.empty_like(x)
y_triton = torch.empty_like(x)

BLOCK_SIZE = 1024
grid = (triton.cdiv(N, BLOCK_SIZE),)

for _ in range(3):  # warm-up both
    torch.add(x, 1.0, out=y_torch)
    add_one_kernel[grid](x, y_triton, N, BLOCK_SIZE=BLOCK_SIZE)
torch.cuda.synchronize()

t_torch = []
for _ in range(10):
    torch.cuda.synchronize()
    start = time.perf_counter()
    torch.add(x, 1.0, out=y_torch)
    torch.cuda.synchronize()
    t_torch.append(time.perf_counter() - start)

t_triton = []
for _ in range(10):
    torch.cuda.synchronize()
    start = time.perf_counter()
    add_one_kernel[grid](x, y_triton, N, BLOCK_SIZE=BLOCK_SIZE)
    torch.cuda.synchronize()
    t_triton.append(time.perf_counter() - start)

bytes_moved = 2 * N * 4
bw_torch = bytes_moved / min(t_torch) / 1e9
bw_triton = bytes_moved / min(t_triton) / 1e9

print(f"torch.add:  {min(t_torch)*1000:.3f} ms -> {bw_torch:.1f} GB/s")
print(f"triton add: {min(t_triton)*1000:.3f} ms -> {bw_triton:.1f} GB/s")
print(f"Triton / torch ratio: {bw_triton/bw_torch:.2f}")
print("Results match:", torch.equal(y_torch, y_triton))

torch.add:  4.501 ms -> 238.5 GB/s
triton add: 4.510 ms -> 238.1 GB/s
Triton / torch ratio: 1.00
Results match: True


## Lesson 27: Fused LayerNorm Kernel

**Goal:** We are building a Fused LayerNorm kernel in Triton. By doing the mean, variance, and normalization in a single GPU pass, we save a lot of time because we don't have to write intermediate results back to VRAM.

In [27]:
@triton.jit
def layernorm_kernel(x_ptr, weight_ptr, bias_ptr, out_ptr, n_cols, eps, BLOCK_SIZE: tl.constexpr):
    row = tl.program_id(0)
    cols = tl.arange(0, BLOCK_SIZE)
    mask = cols < n_cols

    x = tl.load(x_ptr + row * n_cols + cols, mask=mask, other=0.0).to(tl.float32)
    mean = tl.sum(x, axis=0) / n_cols
    diff = tl.where(mask, x - mean, 0.0)
    var = tl.sum(diff * diff, axis=0) / n_cols
    x_norm = (x - mean) / tl.sqrt(var + eps)

    weight = tl.load(weight_ptr + cols, mask=mask, other=1.0).to(tl.float32)
    bias = tl.load(bias_ptr + cols, mask=mask, other=0.0).to(tl.float32)
    y = x_norm * weight + bias
    tl.store(out_ptr + row * n_cols + cols, y, mask=mask)

def triton_layernorm(x, weight, bias, eps):
    n_rows, n_cols = x.shape
    out = torch.empty_like(x)
    BLOCK_SIZE = triton.next_power_of_2(n_cols)
    layernorm_kernel[(n_rows,)](x, weight, bias, out, n_cols, eps, BLOCK_SIZE=BLOCK_SIZE)
    return out

ln1_triton = triton_layernorm(x_manual, blocks[0]["ln_1.weight"], blocks[0]["ln_1.bias"], eps)

print("Shape:", tuple(ln1_triton.shape))
print("Max abs diff vs manual_layernorm (Lesson 14):", (ln1_triton - ln1_manual).abs().max().item())
print("Max abs diff vs HF ln_1:", (ln1_triton - ln1_ref).abs().max().item())

Shape: (5, 768)
Max abs diff vs manual_layernorm (Lesson 14): 5.960464477539063e-08
Max abs diff vs HF ln_1: 8.940696716308594e-08


## Lesson 28: Manual vs Fused LayerNorm Speed

**Goal:** Let's compare our custom Fused LayerNorm with the manual one. Since it reduces memory reads and writes significantly, we should see a huge speedup in isolation.

In [28]:
import time

def bench(fn, warmup=5, repeats=30):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    times = []
    for _ in range(repeats):
        torch.cuda.synchronize()
        start = time.perf_counter()
        fn()
        torch.cuda.synchronize()
        times.append(time.perf_counter() - start)
    return min(times)

w, b = blocks[0]["ln_1.weight"], blocks[0]["ln_1.bias"]
t_manual = bench(lambda: manual_layernorm(x_manual, w, b, eps))
t_triton = bench(lambda: triton_layernorm(x_manual, w, b, eps))

print(f"Manual (multi-op):  {t_manual*1000:.4f} ms")
print(f"Triton (fused):     {t_triton*1000:.4f} ms")
print(f"Speedup: {t_manual/t_triton:.2f}x")

Manual (multi-op):  0.1055 ms
Triton (fused):     0.0410 ms
Speedup: 2.57x


## Lesson 29: Fused LayerNorm in the Full Forward Pass

**Goal:** We are plugging the Triton LayerNorm into our full GPT-2 forward pass. We will see Amdahl's Law in action here: a massive speedup in one layer only gives a small speedup overall because matmuls still take most of the time.

In [29]:
def transformer_block_fused(x, block, n_head, eps):
    ln1 = triton_layernorm(x, block["ln_1.weight"], block["ln_1.bias"], eps)
    qkv = ln1 @ block["attn.c_attn.weight"] + block["attn.c_attn.bias"]
    q, k, v = (split_heads(t, n_head) for t in qkv.split(768, dim=-1))
    attn_out, _ = manual_attention(q, k, v)
    attn_out = merge_heads(attn_out) @ block["attn.c_proj.weight"] + block["attn.c_proj.bias"]
    x = x + attn_out

    ln2 = triton_layernorm(x, block["ln_2.weight"], block["ln_2.bias"], eps)
    h = torch.nn.functional.gelu(ln2 @ block["mlp.c_fc.weight"] + block["mlp.c_fc.bias"], approximate="tanh")
    mlp_out = h @ block["mlp.c_proj.weight"] + block["mlp.c_proj.bias"]
    return x + mlp_out

def forward_full_fused(ids):
    positions = torch.arange(ids.shape[0], device="cuda")
    x = globals_["wte.weight"][ids] + globals_["wpe.weight"][positions]
    for block in blocks:
        x = transformer_block_fused(x, block, n_head, eps)
    x = triton_layernorm(x, globals_["ln_f.weight"], globals_["ln_f.bias"], eps)
    return x @ globals_["wte.weight"].T

t_orig = bench(lambda: forward_full(ids))
t_fused = bench(lambda: forward_full_fused(ids))
diff = (forward_full_fused(ids) - logits_manual).abs().max().item()

print(f"Original forward: {t_orig*1000:.3f} ms")
print(f"Fused-LN forward:  {t_fused*1000:.3f} ms")
print(f"Speedup: {t_orig/t_fused:.2f}x")
print(f"Max abs diff vs Lesson 19 logits: {diff}")

Original forward: 9.495 ms
Fused-LN forward:  8.631 ms
Speedup: 1.10x
Max abs diff vs Lesson 19 logits: 8.392333984375e-05


## Sprint 4 so far
- Triton basics: program_id, BLOCK_SIZE, tl.load/store, mask
- Simple elementwise kernel matched torch's bandwidth exactly (1.00x) , no room to beat an already memory-bound op
- Fused LayerNorm: 2.47x faster in isolation, exact-to-1-ULP correctness (2^-24)
- Full pipeline with fused LN: only 1.13x , LayerNorm is a minority of total launch overhead
- Next target: attention (the biggest, most-repeated multi-step operation)